In [ ]:
import numpy as np
from statsmodels.stats.contingency_tables import StratifiedTable

# Each key = (conference, year)
data = {
    ("NeurIPS", 2023): {"P_USA": 42, "N_USA": 100, "P_CHN": 35, "N_CHN": 100},
    ("NeurIPS", 2024): {"P_USA": 45, "N_USA": 110, "P_CHN": 40, "N_CHN": 100},
    ("NeurIPS", 2025): {"P_USA": 48, "N_USA": 120, "P_CHN": 44, "N_CHN": 105},

    ("ICLR", 2023): {"P_USA": 50, "N_USA": 130, "P_CHN": 30, "N_CHN": 100},
    ("ICLR", 2024): {"P_USA": 55, "N_USA": 140, "P_CHN": 38, "N_CHN": 110},
    ("ICLR", 2025): {"P_USA": 60, "N_USA": 150, "P_CHN": 45, "N_CHN": 120},

    ("ICML", 2023): {"P_USA": 40, "N_USA": 90, "P_CHN": 29, "N_CHN": 80},
    ("ICML", 2024): {"P_USA": 52, "N_USA": 120, "P_CHN": 62, "N_CHN": 140},
    ("ICML", 2025): {"P_USA": 54, "N_USA": 130, "P_CHN": 51, "N_CHN": 125},
}

# Build stratified tables
tables = []

for (conf, year), d in data.items():
    table = np.array([
        [d["P_USA"], d["N_USA"] - d["P_USA"]],
        [d["P_CHN"], d["N_CHN"] - d["P_CHN"]],
    ])
    tables.append(table)

print(tables[0])
# CMH test
st = StratifiedTable(tables)

print("Common odds ratio (MH):", st.oddsratio_pooled)
print("95% CI:", st.oddsratio_pooled_confint())
print("CMH p-value:", st.test_null_odds().pvalue)

: 

🎯 1. What your CMH test is doing

Your code is estimating:

Is there a consistent difference between USA and China in preprint usage, after controlling for conference and year?

Each table = one (conference, year) stratum CMH combines them into one overall effect

📊 2. What each output means 🔹 (A) Common odds ratio (MH) Common odds ratio (MH): X Interpretation:

1 → USA has higher preprint ratio < 1 → China has higher preprint ratio = 1 → no difference Example interpretation

If:

odds ratio = 1.25

👉 Interpretation:

Papers from USA have 25% higher odds of having preprints than China (after controlling for conference and year)

🔹 (B) Confidence interval 95% CI: [low, high] Interpretation: If CI excludes 1 → statistically significant If CI includes 1 → not significant Example CI: [1.10, 1.42]

👉 Strong evidence of difference

CI: [0.95, 1.20]

👉 Not significant

🔹 (C) CMH p-value CMH p-value: p Interpretation: p < 0.05 → significant difference p ≥ 0.05 → no strong evidence 🧠 3. What your specific dataset implies (intuition)

Looking at your data:

USA ratios are generally slightly higher Differences are consistent across strata No reversal (important!)

👉 So you will likely get:

odds ratio > 1 p-value < 0.05 ⚖️ 4. What this means scientifically

You can conclude:

“USA authors are more likely to preprint than China authors, even after controlling for conference and year.”

In [4]:
import numpy as np
from statsmodels.stats.contingency_tables import StratifiedTable

# Example: USA only
data = {
    ("NeurIPS", 2023): {"P_top": 42, "N_top": 100, "P_bot": 25, "N_bot": 100},
    ("NeurIPS", 2024): {"P_top": 50, "N_top": 110, "P_bot": 30, "N_bot": 105},
    ("ICLR", 2023): {"P_top": 55, "N_top": 130, "P_bot": 35, "N_bot": 120},
    ("ICML", 2023): {"P_top": 40, "N_top": 90, "P_bot": 28, "N_bot": 85},
}

tables = []

for (conf, year), d in data.items():
    table = np.array([
        [d["P_top"], d["N_top"] - d["P_top"]],
        [d["P_bot"], d["N_bot"] - d["P_bot"]],
    ])
    tables.append(table)

st = StratifiedTable(tables)

print("Odds Ratio:", st.oddsratio_pooled)
print("95% CI:", st.oddsratio_pooled_confint())
print("p-value:", st.test_null_odds().pvalue)

Odds Ratio: 1.902592313122216
95% CI: (np.float64(1.4284874498004478), np.float64(2.534049221403673))
p-value: 1.014885550176814e-05


🎯 Your Result Odds Ratio (OR): 1.90 95% CI: [1.43, 2.53] p-value: 1.01 × 10⁻⁵ 🧠 1. Core Interpretation

Top-tier institutions have ~90% higher odds of preprinting compared to bottom-tier institutions, after controlling for conference and year.

📊 2. What each number means 🔹 Odds Ratio = 1.90 OR > 1 → Top > Bottom

👉 Specifically:

(odds of preprint in Top) ≈ 1.9 × (odds in Bottom)

⚠️ Important:

This is odds, not raw probability Still very strong effect 🔹 Confidence Interval = [1.43, 2.53] Entire interval > 1 → effect is statistically significant Lower bound (1.43) still large → robust effect

👉 Even in worst case:

Top still ~43% higher odds 🔹 p-value = 1e-5 p << 0.05 → extremely significant

👉 Strong evidence against null hypothesis (no difference)

🔥 3. Scientific Interpretation

You can confidently say:

“Top-tier institutions exhibit substantially higher preprint adoption than bottom-tier institutions.”

⚠️ 4. What this does NOT mean

Be careful — reviewers will check this:

❌ Does NOT mean:

preprints cause acceptance top institutions are “better” because of preprints causal mechanism

👉 This is:

association, not causation 📈 5. Intuition (simple)

If:

Bottom: 30% preprint Top: ~45–50% preprint

👉 Odds ratio ≈ 1.8–2.0 → matches your result

🧾 6. Paper-ready sentence (use this)

“We find that top-tier institutions have significantly higher preprint adoption than bottom-tier institutions (OR = 1.90, 95% CI [1.43, 2.53], p < 0.001), after controlling for conference and year.”

🔍 7. Extra (recommended)

Run this check:

st.test_equal_odds()

👉 If:

p ≥ 0.05 → effect consistent across conferences/years ✅ p < 0.05 → effect varies → report per-stratum 📊 8. How strong is this result? OR Strength 1.1–1.2 weak 1.2–1.5 moderate 1.5–2.0 strong ✅

2.0 very strong

👉 Your result = strong effect

In [6]:
import numpy as np
from statsmodels.stats.contingency_tables import StratifiedTable

# Example data: (conference, year)
data = {
    ("NeurIPS", 2023): {"P_early": 50, "N_early": 100, "P_late": 30, "N_late": 100},
    ("NeurIPS", 2024): {"P_early": 55, "N_early": 110, "P_late": 35, "N_late": 105},
    ("ICLR", 2023): {"P_early": 60, "N_early": 130, "P_late": 40, "N_late": 120},
    ("ICML", 2023): {"P_early": 45, "N_early": 90, "P_late": 28, "N_late": 85},
}

tables = []

for (conf, year), d in data.items():
    table = np.array([
        [d["P_early"], d["N_early"] - d["P_early"]],
        [d["P_late"], d["N_late"] - d["P_late"]],
    ])
    tables.append(table)

st = StratifiedTable(tables)

print("Odds Ratio:", st.oddsratio_pooled)
print("95% CI:", st.oddsratio_pooled_confint())
print("p-value:", st.test_null_odds().pvalue)

Odds Ratio: 1.9872937498212735
95% CI: (np.float64(1.5017555959585793), np.float64(2.6298130392900676))
p-value: 1.4200533375863245e-06


🎯 Your Result Odds Ratio (OR): 1.99 95% CI: [1.50, 2.63] p-value: 1.42 × 10⁻⁶ 🧠 1. Core Interpretation

Papers with preprints posted ≥30 days before deadline have ~2× higher odds compared to those posted <30 days, after controlling for conference and year.

📊 2. What this means intuitively OR ≈ 2.0 Early preprints ≈ 2× as likely as late preprints

👉 This is a very large effect

🔹 Confidence Interval [1.50, 2.63] Entire interval > 1 → statistically significant Even the lower bound (1.50) is strong

👉 Effect is robust

🔹 p-value 1.4e-6 << 0.05

👉 Extremely strong evidence 👉 Not due to chance

🔥 3. Scientific interpretation

You can confidently say:

“Early preprints (≥30 days before deadline) are substantially more prevalent than late preprints across conferences and years.”

⚠️ 4. VERY IMPORTANT clarification

Your setup is:

accepted papers only

So you are measuring:

distribution of preprint timing among accepted papers

NOT:

effect of timing on acceptance

👉 Reviewers will check this — be explicit.

📈 5. How strong is this effect? OR Interpretation 1.1–1.3 weak 1.3–1.6 moderate 1.6–2.0 strong ✅ ~2.0 very strong 🔥

👉 Your result = very strong effect

🧾 6. Paper-ready sentence (use this)

“We find that preprints posted at least 30 days prior to submission are significantly more prevalent than late preprints (OR = 1.99, 95% CI [1.50, 2.63], p < 0.001), controlling for conference and year.”